In [1]:
import os
os.environ["HF_HOME"] = "/workspaces/project/.cache/huggingface"
os.environ["MPLCONFIGDIR"] = "/workspaces/project/.cache/matplotlib" 

from transformers import AutoModelForMaskedLM, AutoTokenizer
import anndata as ad
import numpy as np


/workspaces/project/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [35]:

# Load model and tokenizer
model = AutoModelForMaskedLM.from_pretrained("aletlvl/Nicheformer", trust_remote_code=True)
tokenizer = AutoTokenizer.from_pretrained("aletlvl/Nicheformer", trust_remote_code=True)

# Set technology mean for HF tokenizer
technology_mean_path = '/workspaces/project/src/nicheformer/data/model_means/cosmx_mean_script.npy'
technology_mean = np.load(technology_mean_path)
tokenizer._load_technology_mean(technology_mean)


Using provided technology mean array with shape (20310,)


In [36]:

# Load your single-cell data
adata = ad.read_h5ad("/workspaces/project/data/spe_converted.h5ad")

import scipy.sparse as sp
adata.X = sp.csr_matrix(adata.layers["counts"])

import mygene
import numpy as np

mg = mygene.MyGeneInfo()
res = mg.querymany(
    adata.var_names.tolist(),
    scopes="symbol",          # also try "symbol,alias" for deprecated names
    fields="ensembl.gene",
    species="human",          # ENSMUSG... IDs
    as_dataframe=True,
)

# Collapse duplicates (one symbol can return multiple hits) and pick the first Ensembl ID
res = res[~res.index.duplicated(keep="first")]

# Some symbols return a list under 'ensembl' instead of a scalar 'ensembl.gene'
def pick(row):
    if isinstance(row.get("ensembl.gene"), str):
        return row["ensembl.gene"]
    ens = row.get("ensembl")
    if isinstance(ens, list) and len(ens):
        return ens[0]["gene"]
    return np.nan

ensembl_ids = res.apply(pick, axis=1)

# Stash the symbol, then replace var_names
adata.var["symbol"] = adata.var_names
adata.var["ensembl_id"] = adata.var_names.map(ensembl_ids)

# Drop unmapped, then drop duplicates (multiple symbols → same ENSMUSG)
adata = adata[:, adata.var["ensembl_id"].notna()].copy()
adata = adata[:, ~adata.var["ensembl_id"].duplicated()].copy()
adata.var_names = adata.var["ensembl_id"].values
adata.var_names_make_unique()

print(f"Mapped {adata.n_vars} genes to Ensembl IDs")
print(adata.var_names[:10])  # should now look like ENSMUSG00000...

1 input query terms found dup hits:	[('Cast', 2)]
38 input query terms found no hit:	['Aph1b/c', 'Bex1/2', 'Man1a', 'C1s1/2', 'H2-Aa', '6330403K07Rik', 'Atpif1', 'Ugt8a', 'Car8', 'Pirb'


Mapped 922 genes to Ensembl IDs
Index(['ENSG00000101204', 'ENSG00000157103', 'ENSG00000156535',
       'ENSG00000134333', 'ENSG00000109107', 'ENSG00000184845',
       'ENSG00000136560', 'ENSG00000152214', 'ENSG00000106617',
       'ENSG00000171517'],
      dtype='object')


In [40]:
import torch
import os

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

#device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device = torch.device("cpu")
print(f"Using device: {device}")

torch.cuda.empty_cache()

model = model.half().to(device)
model.eval()

# Tokenize
inputs = tokenizer(adata)

# Truncate sequence length to fit in GPU memory
max_len = 650 + 3
input_ids = inputs["input_ids"][:, :max_len]
attention_mask = inputs["attention_mask"][:, :max_len]

# Patch the positional index to match truncated length
model.nicheformer.pos = model.nicheformer.pos[:max_len]

# Process in batches
batch_size = 64
all_embeddings = []

with torch.no_grad():
    for i in range(0, len(input_ids), batch_size):
        batch_input_ids = input_ids[i:i+batch_size].to(device)
        batch_attention_mask = attention_mask[i:i+batch_size].to(device)

        emb = model.get_embeddings(
            input_ids=batch_input_ids,
            attention_mask=batch_attention_mask,
            layer=-1,
            with_context=True
        )
        all_embeddings.append(emb.cpu())
        torch.cuda.empty_cache()

embeddings = torch.cat(all_embeddings, dim=0)
print(f"Embeddings shape: {embeddings.shape}")

Using device: cpu
Embeddings shape: torch.Size([100, 512])
